# TWFE Robustness: Occupation-Level Clustering

The baseline TWFE specification (`did_unified.ipynb`) clusters standard errors
at the occupation-industry level (`cluster_entity=True`, where the entity is
`OCC_CODE + NAICS`). Because the treatment variable `A_o` (AI exposure) varies
only at the occupation level, residual correlation across industries within
the same occupation is not addressed by occupation-industry clustering, and
the baseline standard errors may be too small.

This notebook re-estimates the pooled TWFE specification (Equation 2 /
`did_pooled_summary.csv`) with standard errors clustered at the occupation
level instead, and reports both side by side.

**Note:** this only reruns the pooled specification (Table 1), not the
event-study. If the pooled coefficients and significance survive, the
event-study conclusions are very unlikely to be overturned by the same
clustering change, since they use an identical panel and estimator.


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from linearmodels.panel import PanelOLS

df = pd.read_csv('bls_onet_felten_ms_panel_harmonised.csv')
df = df[~df['year'].isin([2020, 2021])].copy()

df['TOT_EMP']  = pd.to_numeric(df['TOT_EMP'], errors='coerce')
df['A_MEDIAN'] = pd.to_numeric(df['A_MEDIAN'], errors='coerce')
df['log_emp']  = np.log(df['TOT_EMP'].replace(0, np.nan))
df['log_wage'] = np.log(df['A_MEDIAN'].replace(0, np.nan))

# Post-treatment dummy: 1 if year >= 2023 (2022 is still pre, OEWS is a May survey)
df['post'] = (df['year'] >= 2023).astype(int)

print(f'Base panel rows (before per-index filtering): {len(df)}')


In [ ]:
INDICES = [
    {'col': 'LM_AIOE',  'std_col': 'aioe_std', 'label': 'LM_AIOE'},
    {'col': 'MS_SCORE', 'std_col': 'ms_std',   'label': 'MS_SCORE'},
]
OUTCOMES = ['log_emp', 'log_wage']

def prep_index_panel(base_df, idx_col, std_col):
    """Filter to rows with this index available, standardise, build did +
    unit index. OCC_CODE is kept as a plain column so it can be used as the
    clustering variable below (it is coarser than the occupation-industry
    entity index)."""
    d = base_df[base_df[idx_col].notna()].copy()
    d[std_col] = (d[idx_col] - d[idx_col].mean()) / d[idx_col].std()
    d['did'] = d[std_col] * d['post']
    d['unit'] = d['OCC_CODE'] + '_' + d['NAICS'].astype(str)
    return d

panels = {ix['label']: prep_index_panel(df, ix['col'], ix['std_col']) for ix in INDICES}
for label, d in panels.items():
    print(f'{label}: {len(d)} rows, {d["OCC_CODE"].nunique()} occupations')


## Re-estimate the pooled specification under both clustering choices

For each (index, outcome) combination this fits the identical model twice:

1. **Occupation-industry clustering** (`cluster_entity=True`) -- the baseline
   reported in the dissertation, clustering at the level of the panel entity.
2. **Occupation clustering** (`clusters=OCC_CODE`) -- the coarser level at
   which the treatment variable actually varies.


In [ ]:
comparison_rows = []

for ix in INDICES:
    label = ix['label']
    d = panels[label].set_index(['unit', 'year'])
    for outcome in OUTCOMES:
        sample = d[[outcome, 'did', 'OCC_CODE']].dropna()

        # --- Baseline: occupation-industry (entity) clustering ---
        model = PanelOLS.from_formula(f'{outcome} ~ did + EntityEffects + TimeEffects',
                                       data=sample[[outcome, 'did']])
        res_entity = model.fit(cov_type='clustered', cluster_entity=True)

        # --- Robustness: occupation-level clustering ---
        # clusters must be a Series aligned to the model's data, giving each
        # row's cluster id; here that id is the occupation code, which is
        # coarser than the occupation-industry entity index above.
        res_occ = model.fit(cov_type='clustered', clusters=sample['OCC_CODE'])

        print(f'=== {label} x {outcome} ===')
        print(f'  Occupation-industry clustering: coef={res_entity.params["did"]:.4f}, '
              f'SE={res_entity.std_errors["did"]:.4f}, p={res_entity.pvalues["did"]:.4g}, '
              f'n_clusters={sample.index.get_level_values("unit").nunique()}')
        print(f'  Occupation-level clustering:     coef={res_occ.params["did"]:.4f}, '
              f'SE={res_occ.std_errors["did"]:.4f}, p={res_occ.pvalues["did"]:.4g}, '
              f'n_clusters={sample["OCC_CODE"].nunique()}')
        print()

        comparison_rows.append({
            'index': label,
            'outcome': outcome,
            'coef': res_entity.params['did'],
            'se_occ_industry_cluster': res_entity.std_errors['did'],
            'pvalue_occ_industry_cluster': res_entity.pvalues['did'],
            'n_clusters_occ_industry': sample.index.get_level_values('unit').nunique(),
            'se_occupation_cluster': res_occ.std_errors['did'],
            'pvalue_occupation_cluster': res_occ.pvalues['did'],
            'n_clusters_occupation': sample['OCC_CODE'].nunique(),
            'n_obs': res_entity.nobs,
        })


## Summary table


In [ ]:
comparison = pd.DataFrame(comparison_rows)
comparison['se_ratio'] = comparison['se_occupation_cluster'] / comparison['se_occ_industry_cluster']
comparison.to_csv('did_occupation_clustering_comparison.csv', index=False)

pd.set_option('display.width', 140)
print(comparison.to_string(index=False))
print()
print('se_ratio > 1 means occupation-level clustering gives a larger (more conservative) SE than the baseline.')
